## 类创建对象，元类创建类

在 Python 中“万物皆对象”，类本身也是一个对象。普通类是创建“实例对象”的蓝图（Obj = Class()），而元类（Metaclass）就是创建“类对象”的蓝图（Class = Metaclass()）。

1. 第一层（浅）：type 的双重身份

Python 中内置的 type 不仅能检查数据类型，它本身就是所有默认类的“元类”。

用 class 关键字定义类时，Python 底层会自动调用 type 动态创建它：

In [3]:
# 1. 常见写法
# class Dog:
#     legs = 4

# 2. type 动态创建等价写法：type(类名, 父类元组, 属性与方法字典)
Dog = type('Dog', (), {'legs': 4})

print(type(Dog))  # <class 'type'> -> Dog 这个类对象是由 type 生成的

<class 'type'>


2. 第二层（中）：自定义元类与属性拦截

自定义元类必须继承自 type，通过重写 __new__ 方法，可以在类被创建（加载）的那一刻拦截并修改类的结构。

In [5]:
class UpperAttrMeta(type):
    # __new__ 在类对象分配内存前触发
    def __new__(mcls, name, bases, attrs):
        # 拦截所有非私有属性，将其名称转为大写
        uppercase_attrs = {
            k if k.startswith('__') else k.upper(): v
            for k, v in attrs.items()
        }
        return super().__new__(mcls, name, bases, uppercase_attrs)

# 指定 metaclass
class Foo(metaclass=UpperAttrMeta):
    bar = 'bip'

print(hasattr(Foo, 'bar'))  # False
# print(Foo.bar) # 抛出异常
print(Foo.BAR)              # 'bip'

False
bip


3. 第三层（深）：控制实例化过程（__call__ 与单例模式）

__new__：控制类对象的创建（定义类时运行一次）。

__call__：控制实例对象的创建（每次执行 obj = MyClass() 时运行）。

通过重写元类的 __call__，可以轻松拦截并实现单例模式：


In [10]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        # 当执行 Database() 时触发
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Database(metaclass=SingletonMeta):
    pass

db1 = Database()
db2 = Database()
print(db1 is db2)  # True，始终返回同一个实例

True


单例模式中代码的执行逻辑拆解

In [ ]:
# 当你第一次执行 db1 = Database() 时：
def __call__(cls, *args, **kwargs): # cls 就是 Database 这个类
    if cls not in cls._instances:   # 检查 Database 是否已经在字典 Key 里
        # 1. 触发真正的类实例化，创建出 Database 的实例对象
        new_instance = super().__call__(*args, **kwargs)
        # 2. 将 { Database类 : Database实例 } 存入字典
        cls._instances[cls] = new_instance
    # 3. 返回存好的实例
    return cls._instances[cls]

4. 现代 Python 替代方案与使用原则

框架开发：元类常用于 ORM（如 Django/SQLAlchemy，将类属性映射为数据库列）或复杂插件系统。

轻量替代方案：Python 3.6+ 引入了 __init_subclass__ 钩子函数，不需要定义元类即可在基类中监控/修改子类创建，更加简单直观。

开发原则：正如 Tim Peters 所说：“如果你在犹豫是否需要使用元类，那你其实就不需要。”大多数日常业务逻辑，用装饰器或继承即可解决。

## 元类创建的类对象本身是一个单例吗

不是绝对的，这取决于从哪个层面来看：

在常规代码定义中，通过 `class` 关键字创建的类对象在内存中是唯一的（具备单例特性）；但从底层机制来看，元类本身并不限制创建多个独立的类对象。

### 常规定义下（像单例）：
当 Python 解析并执行 `class Foo: pass` 代码块时，元类（默认是 `type`）只会运行一次来创建 `Foo` 这个类对象。后续在代码任意位置引用 `Foo`，指向的都是内存中的同一个地址：

In [ ]:
class Foo: pass

A = Foo
B = Foo
print(A is B)  # True，指向同一个类对象

### 动态调用下（不是单例）：
元类的本质是“生成类的工厂”。如果多次调用元类，每次都会生成一个全新的、占用独立内存的类对象，即使它们同名或结构完全相同：

In [ ]:
# 显式调用元类 type 动态创建类
Class1 = type('Foo', (), {})
Class2 = type('Foo', (), {})

print(Class1 == Class2) # False
print(Class1 is Class2) # False（它们是两个不同的类对象）

### 补充：区分“类对象的单例”与“实例对象的单例”

在日常开发面试中，提到“元类”与“单例”时，通常是指用元类去控制普通类只能产生一个实例对象（即限制 obj1 = MyClass() 和 obj2 = MyClass() 返回同一个实例）：

```Python
class SingletonMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        # 控制该类生成的实例是单例
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Database(metaclass=SingletonMeta):
    pass

db1 = Database()
db2 = Database()
print(db1 is db2)  # True
总结：元类创建出来的类对象，在模块加载时是天然全局唯一的；但元类本身并不禁止再次调用它去产生新的类对象。
```